In [5]:
!brew services start ollama

==> Successfully started `ollama` (label: homebrew.mxcl.ollama)


In [7]:
!ollama pull all-minilm

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 797b70c4edf8: 100% ▕██████████████████▏  45 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling 85011998c600: 100% ▕██████████████████▏   16 B                         
pulling 548455b72658: 100% ▕██████████████████▏  407 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
import sys
import ollama
import numpy as np

# -------------------------------------------------------------------------
# 1. THE IN-MEMORY KNOWLEDGE BASE
# Replace these strings with your own custom or proprietary documentation snippets.
# -------------------------------------------------------------------------
KNOWLEDGE_BASE = [
    "Studio Ghibli was co-founded by Hayao Miyazaki and Isao Takahata in 1985. Its breakout masterpiece 'Spirited Away' (2001) won the Academy Award for Best Animated Feature.",
    "The term 'Cyberpunk' in anime was heavily defined by Katsuhiro Otomo's 1988 film 'Akira' and Mamoru Oshii's 1995 tactical sci-fi film 'Ghost in the Shell'.",
    "Neon Genesis Evangelion, directed by Hideaki Anno and produced by Studio Gainax in 1995, revolutionized the mecha genre by focusing heavily on psychological trauma and existential philosophy.",
    "The term 'Shonen' refers to anime targeted primarily at young teenage boys, characterized by action-packed plots and themes of camaraderie, exemplified by 'Dragon Ball', 'Naruto', and 'One Piece'.",
    "Studio Ufotable is widely acclaimed for its cutting-edge digital animation techniques and dynamic camera work, best seen in their production of 'Demon Slayer: Kimetsu no Yaiba'.",
    "The 'Iyashikei' (healing) genre is a sub-category of slice-of-life anime designed to have a calming, therapeutic effect on the audience, featuring peaceful environments like 'Yuru Camp'."]

def get_embedding(text: str, model: str = "all-minilm") -> list:
    """Generates a high-dimensional vector representation using a dedicated embedding model."""
    try:
        # Note: Depending on your Ollama SDK version, use ollama.embeddings() or ollama.embed()
        response = ollama.embeddings(model=model, prompt=text)
        return response['embedding']
    except Exception as e:
        print(f"\n[Error generating embedding]: Ensure ollama is running. Detail: {e}")
        sys.exit(1)

def cosine_similarity(v1, v2) -> float:
    """Calculates the angular distance between two vectors to evaluate conceptual match."""
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

# Pre-compute embeddings for our documents up front to optimize search runtime speed
print("--> Initializing knowledge base embeddings...")
DOCUMENT_EMBEDDINGS = [get_embedding(doc) for doc in KNOWLEDGE_BASE]
print("--> Vector indexing complete. Ready for queries.\n")

# -------------------------------------------------------------------------
# 2. THE RETRIEVAL COMPONENT
# -------------------------------------------------------------------------
def retrieve_relevant_context(query: str, top_k: int = 1) -> str:
    """Compares user query vector against text database and extracts matching contexts."""
    query_vector = get_embedding(query)
    similarities = []

    for idx, doc_vector in enumerate(DOCUMENT_EMBEDDINGS):
        score = cosine_similarity(query_vector, doc_vector)
        similarities.append((score, KNOWLEDGE_BASE[idx]))
    
    # Sort by descending similarity score
    similarities.sort(key=lambda x: x[0], reverse=True)
    
    # Grab the top-K matches and fuse them into a cohesive block
    retrieved_snippets = [item[1] for item in similarities[:top_k]]
    return "\n".join(retrieved_snippets)

# -------------------------------------------------------------------------
# 3. INTERACTIVE CONVERSATION LOOP
# -------------------------------------------------------------------------
def run_chat_session():
    # Maintain conversational memory state using Ollama's chat array scheme
    session_history = []
    
    print("=====================================================================")
    print("      Gemma-2:2b RAG Terminal Chatbot (Type 'exit' or 'quit' to close)")
    print("=====================================================================")
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
            if not user_input:
                continue
            if user_input.lower() in ['exit', 'quit']:
                print("Ending session. Goodbye.")
                break
            
            # Phase A: Query Vector Database for real-time context
            context = retrieve_relevant_context(user_input, top_k=2)
            
            # Phase B: Engineering the Prompt
            # We explicitly instruct the model to fail safe if the context is missing info.
            system_instruction = (
                "You are an elite enterprise security assistant. Answer the user's question "
                "using ONLY the provided context snippets below. If the context does not contain "
                "the answers or data required to address the prompt, state cleanly: "
                "'I cannot find that information in my data source.'\n\n"
                f"--- CONTEXT REFERENCE BLOCK ---\n{context}\n-------------------------------"
            )
            
            # Cleanly reconstruct current message array ensuring proper Gemma syntax targeting
            messages = [{"role": "system", "content": system_instruction}]
            
            # Append historical context back so conversational memory holds together
            messages.extend(session_history)
            messages.append({"role": "user", "content": user_input})
            
            # Phase C: Streaming Inference Response
            print("Gemma-2: ", end="", flush=True)
            
            stream = ollama.chat(
                model="gemma2:2b",
                messages=messages,
                stream=True,
                options={"temperature": 0.1}  # Low temperature forces deterministic adherence to data
            )
            
            full_response_text = ""
            for chunk in stream:
                content = chunk['message']['content']
                print(content, end="", flush=True)
                full_response_text += content
            print()
            
            # Preserve state transitions in memory
            session_history.append({"role": "user", "content": user_input})
            session_history.append({"role": "assistant", "content": full_response_text})
            
        except KeyboardInterrupt:
            print("\nSession interrupted. Exiting.")
            break

if __name__ == "__main__":
    run_chat_session()

--> Initializing knowledge base embeddings...
--> Vector indexing complete. Ready for queries.

      Gemma-2:2b RAG Terminal Chatbot (Type 'exit' or 'quit' to close)



You:  What's Spirited Away?


Gemma-2: 'Spirited Away' (2001) won the Academy Award for Best Animated Feature. 




You:  Who directed it and what year?


Gemma-2: It was directed by Hayao Miyazaki and produced by Studio Ghibli in 2001. 



In [4]:
!brew services stop ollama

Stopping `ollama`... (might take a while)
==> Successfully stopped `ollama` (label: homebrew.mxcl.ollama)
